# Preprocesamiento de Datos - Riesgos Laborales Colombia

Este notebook realiza el preprocesamiento del dataset de **Estadísticas de Riesgos Laborales** para prepararlo para el modelado de Machine Learning.

## Objetivos:
1. Limpiar los datos (manejo de valores faltantes, duplicados, outliers).
2. Transformar variables categóricas y numéricas.
3. Crear variables derivadas (tasas, ratios).
4. Exportar el dataset curado para su uso en los siguientes notebooks.

## 1. Configuración Inicial

### 1.1 Importar Librerías

Importamos las librerías necesarias para el preprocesamiento de datos.

In [15]:
# Instala las librerias de pandas, numpy, matplotlib, seaborn, scikit-learn y joblib
!pip install pandas numpy matplotlib seaborn scikit-learn joblib

In [2]:
# Importar librerías necesarias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from pathlib import Path
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer
import joblib

# Configuración de visualización
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Configuración de pandas para mostrar todas las columnas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)

print("✅ Librerías importadas correctamente")

✅ Librerías importadas correctamente


### 1.2 Montar Google Drive (para Colab)

Si estás ejecutando este notebook en Google Colab, es necesario montar Google Drive para acceder a los archivos del proyecto.

In [3]:
# Verificar si estamos en Google Colab
try:
    from google.colab import drive
    # Montar Google Drive
    drive.mount('/content/drive')
    # Definir ruta base del proyecto en Google Drive
    BASE_PATH = '/content/drive/MyDrive/DatosARLBog'
    print(f"✅ Google Drive montado correctamente")
    print(f"📁 Ruta base del proyecto: {BASE_PATH}")
except:
    # Si no estamos en Colab, usar la ruta local
    BASE_PATH = '/Users/jualgozo/Documents/datoscol/DatosArl_Mac'
    print(f"💻 Ejecutando en entorno local")
    print(f"📁 Ruta base del proyecto: {BASE_PATH}")

💻 Ejecutando en entorno local
📁 Ruta base del proyecto: /Users/jualgozo/Documents/datoscol/DatosArl_Mac


### 1.3 Definir Rutas de Archivos

Definimos las rutas de los archivos del proyecto, incluyendo el dataset original y los directorios de salida.

In [4]:
# Definir rutas de archivos
DATA_PATH = Path(BASE_PATH)
RAW_DATA_FILE = DATA_PATH / 'main_dataset.csv'
PROCESSED_DATA_DIR = DATA_PATH / 'data' / 'processed'
REPORTS_DIR = DATA_PATH / 'reports'
MODELS_DIR = DATA_PATH / 'models'

# Crear directorios si no existen
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print(f"📂 Directorio de datos: {DATA_PATH}")
print(f"📊 Archivo de datos: {RAW_DATA_FILE}")
print(f"📁 Directorio de datos procesados: {PROCESSED_DATA_DIR}")
print(f"📈 Directorio de reportes: {REPORTS_DIR}")
print(f"🤖 Directorio de modelos: {MODELS_DIR}")

📂 Directorio de datos: /Users/jualgozo/Documents/datoscol/DatosArl_Mac
📊 Archivo de datos: /Users/jualgozo/Documents/datoscol/DatosArl_Mac/main_dataset.csv
📁 Directorio de datos procesados: /Users/jualgozo/Documents/datoscol/DatosArl_Mac/data/processed
📈 Directorio de reportes: /Users/jualgozo/Documents/datoscol/DatosArl_Mac/reports
🤖 Directorio de modelos: /Users/jualgozo/Documents/datoscol/DatosArl_Mac/models


## 2. Carga de Datos

### 2.1 Cargar el Dataset Original

Cargamos el dataset original y realizamos una revisión inicial.

In [5]:
# Cargar el dataset original
try:
    df = pd.read_csv(RAW_DATA_FILE)
    print(f"✅ Dataset cargado correctamente")
    print(f"📊 Dimensiones: {df.shape[0]} filas x {df.shape[1]} columnas")
    print(f"\n📋 Columnas disponibles:")
    print(df.columns.tolist())
except FileNotFoundError:
    print(f"❌ Error: No se encontró el archivo {RAW_DATA_FILE}")
    print("Por favor, verifica que el archivo esté en la ruta correcta.")

✅ Dataset cargado correctamente
📊 Dimensiones: 61368 filas x 14 columnas

📋 Columnas disponibles:
['DPTO', 'MPIO', 'CODIGO_DE_LA_ARL', 'AÑO_DE_INFORME', 'MES_DE_INFORME', 'ACTIVEC', 'RELA_DEP', 'RELA_INDEP', 'PRESUACCIDETRASUCE', 'MUERTES_REPOR_AT', 'NUEVAPENSIOINVA_R_AT', 'NUEVAPENSIOINVA_R_EL', 'INCAPERMAPARCIAR_AT', 'INCAPERMAPARCIAR_EL']


## 3. Limpieza de Datos

### 3.1 Manejo de Valores Faltantes

Identificamos y manejamos los valores faltantes en el dataset.

In [6]:
# Análisis de valores faltantes
print("=" * 80)
print("🔍 ANÁLISIS DE VALORES FALTANTES")
print("=" * 80)

# Calcular valores faltantes
missing_values = df.isnull().sum()
missing_percentage = (missing_values / len(df)) * 100

# Crear DataFrame con resultados
missing_df = pd.DataFrame({
    'Columna': missing_values.index,
    'Valores Faltantes': missing_values.values,
    'Porcentaje (%)': missing_percentage.values
})

# Filtrar solo columnas con valores faltantes
missing_df = missing_df[missing_df['Valores Faltantes'] > 0].sort_values('Porcentaje (%)', ascending=False)

if len(missing_df) > 0:
    print(f"\n❌ Se encontraron {len(missing_df)} columnas con valores faltantes:\n")
    print(missing_df.to_string(index=False))
    
    # Decisión sobre valores faltantes
    print("\n📋 DECISIÓN SOBRE VALORES FALTANTES:")
    print("   - Columnas con < 5% de valores faltantes: Imputar con la moda (categóricas) o mediana (numéricas)")
    print("   - Columnas con > 5% de valores faltantes: Evaluar eliminación o imputación avanzada")
else:
    print("\n✅ No se encontraron valores faltantes en el dataset")

🔍 ANÁLISIS DE VALORES FALTANTES

✅ No se encontraron valores faltantes en el dataset


In [7]:
# Imputar valores faltantes
print("=" * 80)
print("🔧 IMPUTACIÓN DE VALORES FALTANTES")
print("=" * 80)

# Crear una copia del DataFrame para no modificar el original
df_clean = df.copy()

# Identificar columnas numéricas y categóricas
numeric_cols = df_clean.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df_clean.select_dtypes(include=['object']).columns.tolist()

print(f"\n📊 Columnas numéricas: {len(numeric_cols)}")
print(f"📊 Columnas categóricas: {len(categorical_cols)}")

# Imputar valores faltantes en columnas numéricas
if len(numeric_cols) > 0:
    print("\n🔧 Imputando valores faltantes en columnas numéricas...")
    for col in numeric_cols:
        if df_clean[col].isnull().sum() > 0:
            # Usar mediana para imputación
            median_value = df_clean[col].median()
            df_clean[col].fillna(median_value, inplace=True)
            print(f"   - {col}: Imputado con mediana ({median_value})")

# Imputar valores faltantes en columnas categóricas
if len(categorical_cols) > 0:
    print("\n🔧 Imputando valores faltantes en columnas categóricas...")
    for col in categorical_cols:
        if df_clean[col].isnull().sum() > 0:
            # Usar moda para imputación
            mode_value = df_clean[col].mode()[0]
            df_clean[col].fillna(mode_value, inplace=True)
            print(f"   - {col}: Imputado con moda ({mode_value})")

print(f"\n✅ Valores faltantes imputados correctamente")
print(f"📊 Valores faltantes restantes: {df_clean.isnull().sum().sum()}")

🔧 IMPUTACIÓN DE VALORES FALTANTES

📊 Columnas numéricas: 10
📊 Columnas categóricas: 4

🔧 Imputando valores faltantes en columnas numéricas...

🔧 Imputando valores faltantes en columnas categóricas...

✅ Valores faltantes imputados correctamente
📊 Valores faltantes restantes: 0


### 3.2 Manejo de Duplicados

Identificamos y eliminamos filas duplicadas.

In [8]:
# Análisis y eliminación de duplicados
print("=" * 80)
print("🔍 ANÁLISIS DE DUPLICADOS")
print("=" * 80)

# Identificar duplicados
duplicates = df_clean.duplicated().sum()
print(f"\n📊 Total de filas duplicadas: {duplicates}")
print(f"📊 Porcentaje de duplicados: {(duplicates / len(df_clean)) * 100:.2f}%")

if duplicates > 0:
    print("\n🔧 Eliminando filas duplicadas...")
    df_clean = df_clean.drop_duplicates()
    print(f"✅ Filas duplicadas eliminadas")
    print(f"📊 Dimensiones después de eliminar duplicados: {df_clean.shape[0]} filas x {df_clean.shape[1]} columnas")
else:
    print("\n✅ No se encontraron filas duplicadas")

🔍 ANÁLISIS DE DUPLICADOS

📊 Total de filas duplicadas: 0
📊 Porcentaje de duplicados: 0.00%

✅ No se encontraron filas duplicadas


### 3.3 Manejo de Outliers

Identificamos y manejamos outliers en las variables numéricas.

In [9]:
# Análisis de outliers
print("=" * 80)
print("🔍 ANÁLISIS DE OUTLIERS")
print("=" * 80)

# Identificar outliers usando el método IQR
def detect_outliers_iqr(df, column):
    """Detecta outliers usando el método IQR"""
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = df[(df[column] < lower_bound) | (df[column] > upper_bound)]
    return outliers, lower_bound, upper_bound

# Analizar outliers en columnas numéricas clave
key_numeric_cols = ['PRESUACCIDETRASUCE', 'MUERTES_REPOR_AT', 'NUEVAPENSIOINVA_R_AT', 
                    'NUEVAPENSIOINVA_R_EL', 'INCAPERMAPARCIAR_AT', 'INCAPERMAPARCIAR_EL']

print("\n📊 ANÁLISIS DE OUTLIERS EN VARIABLES CLAVE:")
for col in key_numeric_cols:
    if col in df_clean.columns:
        outliers, lower, upper = detect_outliers_iqr(df_clean, col)
        num_outliers = len(outliers)
        pct_outliers = (num_outliers / len(df_clean)) * 100
        print(f"\n   {col}:")
        print(f"   - Outliers: {num_outliers} ({pct_outliers:.2f}%)")
        print(f"   - Límite inferior: {lower:.2f}")
        print(f"   - Límite superior: {upper:.2f}")

print("\n📋 DECISIÓN SOBRE OUTLIERS:")
print("   - Los outliers en este contexto pueden representar casos reales de alto riesgo")
print("   - Se recomienda NO eliminar outliers, pero documentarlos para análisis posterior")
print("   - Se pueden crear variables binarias para identificar outliers si es necesario")

🔍 ANÁLISIS DE OUTLIERS

📊 ANÁLISIS DE OUTLIERS EN VARIABLES CLAVE:

   PRESUACCIDETRASUCE:
   - Outliers: 3091 (5.04%)
   - Límite inferior: 0.00
   - Límite superior: 0.00

   MUERTES_REPOR_AT:
   - Outliers: 3091 (5.04%)
   - Límite inferior: 0.00
   - Límite superior: 0.00

   NUEVAPENSIOINVA_R_AT:
   - Outliers: 3091 (5.04%)
   - Límite inferior: 0.00
   - Límite superior: 0.00

   NUEVAPENSIOINVA_R_EL:
   - Outliers: 14 (0.02%)
   - Límite inferior: 0.00
   - Límite superior: 0.00

   INCAPERMAPARCIAR_AT:
   - Outliers: 9 (0.01%)
   - Límite inferior: 0.00
   - Límite superior: 0.00

   INCAPERMAPARCIAR_EL:
   - Outliers: 2 (0.00%)
   - Límite inferior: 0.00
   - Límite superior: 0.00

📋 DECISIÓN SOBRE OUTLIERS:
   - Los outliers en este contexto pueden representar casos reales de alto riesgo
   - Se recomienda NO eliminar outliers, pero documentarlos para análisis posterior
   - Se pueden crear variables binarias para identificar outliers si es necesario


## 4. Transformación de Variables

### 4.1 Codificación de Variables Categóricas

Codificamos las variables categóricas para su uso en modelos de Machine Learning.

In [10]:
# Codificación de variables categóricas
print("=" * 80)
print("🔧 CODIFICACIÓN DE VARIABLES CATEGÓRICAS")
print("=" * 80)

# Identificar columnas categóricas
categorical_cols = df_clean.select_dtypes(include=['object']).columns.tolist()
print(f"\n📊 Columnas categóricas identificadas: {categorical_cols}")

# Crear diccionario para guardar los encoders
label_encoders = {}

# Aplicar Label Encoding a columnas categóricas
print("\n🔧 Aplicando Label Encoding a variables categóricas...")
for col in categorical_cols:
    le = LabelEncoder()
    df_clean[f'{col}_encoded'] = le.fit_transform(df_clean[col].astype(str))
    label_encoders[col] = le
    
    # Mostrar mapeo
    print(f"\n   {col}:")
    print(f"   - Valores únicos: {df_clean[col].nunique()}")
    print(f"   - Mapeo: {dict(zip(le.classes_, le.transform(le.classes_)))}")

print(f"\n✅ Variables categóricas codificadas correctamente")
print(f"📊 Nuevas columnas creadas: {[f'{col}_encoded' for col in categorical_cols]}")

🔧 CODIFICACIÓN DE VARIABLES CATEGÓRICAS

📊 Columnas categóricas identificadas: ['DPTO', 'MPIO', 'RELA_DEP', 'RELA_INDEP']

🔧 Aplicando Label Encoding a variables categóricas...

   DPTO:
   - Valores únicos: 33
   - Mapeo: {'AMAZONAS': np.int64(0), 'ANTIOQUIA': np.int64(1), 'ARAUCA': np.int64(2), 'ATLANTICO': np.int64(3), 'BOGOTA': np.int64(4), 'BOLIVAR': np.int64(5), 'BOYACA': np.int64(6), 'CALDAS': np.int64(7), 'CAQUETA': np.int64(8), 'CASANARE': np.int64(9), 'CAUCA': np.int64(10), 'CESAR': np.int64(11), 'CHOCO': np.int64(12), 'CORDOBA': np.int64(13), 'CUNDINAMARCA': np.int64(14), 'GUAINIA': np.int64(15), 'GUAVIARE': np.int64(16), 'HUILA': np.int64(17), 'LA GUAJIRA': np.int64(18), 'MAGDALENA': np.int64(19), 'META': np.int64(20), 'N. DE SANTANDER': np.int64(21), 'NARIÃ‘O': np.int64(22), 'PUTUMAYO': np.int64(23), 'QUINDIO': np.int64(24), 'RISARALDA': np.int64(25), 'SAN ANDRES': np.int64(26), 'SANTANDER': np.int64(27), 'SUCRE': np.int64(28), 'TOLIMA': np.int64(29), 'VALLE DEL CAUCA': np

## 5. Creación de Variables Derivadas

### 5.1 Tasas y Ratios

Creamos variables derivadas que pueden ser útiles para el análisis y modelado.

In [11]:
# Crear variables derivadas ANTES del escalado
print("=" * 80)
print("🔧 CREACIÓN DE VARIABLES DERIVADAS")
print("=" * 80)

# Crear una copia del DataFrame para agregar variables derivadas
# Usamos df_clean (antes del escalado) para asegurar que las operaciones aritméticas funcionen
df_final = df_clean.copy()

# Limpiar columnas numéricas que pueden tener formato de string con comas
print("\n🔧 Limpiando formato de columnas numéricas...")
numeric_cols_to_clean = ['RELA_DEP', 'RELA_INDEP', 'PRESUACCIDETRASUCE', 'MUERTES_REPOR_AT', 
                          'NUEVAPENSIOINVA_R_AT', 'NUEVAPENSIOINVA_R_EL', 
                          'INCAPERMAPARCIAR_AT', 'INCAPERMAPARCIAR_EL']

for col in numeric_cols_to_clean:
    if col in df_final.columns:
        # Convertir a string primero, luego reemplazar comas y convertir a float
        df_final[col] = df_final[col].astype(str).str.replace(',', '').astype(float)
        print(f"   - {col}: Limpiado y convertido a float")

# 1. Tasa de accidentes por trabajador dependiente
print("\n🔧 Creando tasa de accidentes por trabajador dependiente...")
df_final['TASA_ACCIDENTES_DEP'] = df_final['PRESUACCIDETRASUCE'] / (df_final['RELA_DEP'] + 1)  # +1 para evitar división por cero

# 2. Tasa de muertes por trabajador dependiente
print("🔧 Creando tasa de muertes por trabajador dependiente...")
df_final['TASA_MUERTES_DEP'] = df_final['MUERTES_REPOR_AT'] / (df_final['RELA_DEP'] + 1)

# 3. Tasa de pensiones por accidente
print("🔧 Creando tasa de pensiones por accidente...")
df_final['TASA_PENSIONES_ACCIDENTE'] = df_final['NUEVAPENSIOINVA_R_AT'] / (df_final['PRESUACCIDETRASUCE'] + 1)

# 4. Tasa de pensiones por enfermedad
print("🔧 Creando tasa de pensiones por enfermedad...")
df_final['TASA_PENSIONES_ENFERMEDAD'] = df_final['NUEVAPENSIOINVA_R_EL'] / (df_final['RELA_DEP'] + df_final['RELA_INDEP'] + 1)

# 5. Total de pensiones
print("🔧 Creando total de pensiones...")
df_final['TOTAL_PENSIONES'] = df_final['NUEVAPENSIOINVA_R_AT'] + df_final['NUEVAPENSIOINVA_R_EL']

# 6. Total de incapacidades
print("🔧 Creando total de incapacidades...")
df_final['TOTAL_INCAPACIDADES'] = df_final['INCAPERMAPARCIAR_AT'] + df_final['INCAPERMAPARCIAR_EL']

# 7. Ratio de trabajadores independientes vs dependientes
print("🔧 Creando ratio de trabajadores independientes vs dependientes...")
df_final['RATIO_INDEP_DEP'] = df_final['RELA_INDEP'] / (df_final['RELA_DEP'] + 1)

# 8. Total de trabajadores
print("🔧 Creando total de trabajadores...")
df_final['TOTAL_TRABAJADORES'] = df_final['RELA_DEP'] + df_final['RELA_INDEP']

print(f"\n✅ Variables derivadas creadas correctamente")
print(f"📊 Nuevas columnas: {['TASA_ACCIDENTES_DEP', 'TASA_MUERTES_DEP', 'TASA_PENSIONES_ACCIDENTE', 'TASA_PENSIONES_ENFERMEDAD', 'TOTAL_PENSIONES', 'TOTAL_INCAPACIDADES', 'RATIO_INDEP_DEP', 'TOTAL_TRABAJADORES']}")

# Mostrar estadísticas de las nuevas variables
print("\n📊 Estadísticas de las nuevas variables:")
print(df_final[['TASA_ACCIDENTES_DEP', 'TASA_MUERTES_DEP', 'TOTAL_PENSIONES', 'TOTAL_INCAPACIDADES']].describe().T)

🔧 CREACIÓN DE VARIABLES DERIVADAS

🔧 Limpiando formato de columnas numéricas...
   - RELA_DEP: Limpiado y convertido a float
   - RELA_INDEP: Limpiado y convertido a float
   - PRESUACCIDETRASUCE: Limpiado y convertido a float
   - MUERTES_REPOR_AT: Limpiado y convertido a float
   - NUEVAPENSIOINVA_R_AT: Limpiado y convertido a float
   - NUEVAPENSIOINVA_R_EL: Limpiado y convertido a float
   - INCAPERMAPARCIAR_AT: Limpiado y convertido a float
   - INCAPERMAPARCIAR_EL: Limpiado y convertido a float

🔧 Creando tasa de accidentes por trabajador dependiente...
🔧 Creando tasa de muertes por trabajador dependiente...
🔧 Creando tasa de pensiones por accidente...
🔧 Creando tasa de pensiones por enfermedad...
🔧 Creando total de pensiones...
🔧 Creando total de incapacidades...
🔧 Creando ratio de trabajadores independientes vs dependientes...
🔧 Creando total de trabajadores...

✅ Variables derivadas creadas correctamente
📊 Nuevas columnas: ['TASA_ACCIDENTES_DEP', 'TASA_MUERTES_DEP', 'TASA_PENS

## 4.2 Escalado de Variables Numéricas

Escalamos las variables numéricas para normalizar su rango.

In [12]:
# Escalado de variables numéricas (DESPUÉS de crear variables derivadas)
print("=" * 80)
print("🔧 ESCALADO DE VARIABLES NUMÉRICAS")
print("=" * 80)

# Identificar columnas numéricas (excluyendo las codificadas)
numeric_cols = df_final.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [col for col in numeric_cols if not col.endswith('_encoded')]

# Variables categóricas/identificadoras que NO deben escalarse
# Estas variables son numéricas pero representan categorías o identificadores
categorical_numeric_cols = ['CODIGO_DE_LA_ARL', 'AÑO_DE_INFORME', 'MES_DE_INFORME', 'ACTIVEC']

# Filtrar variables categóricas/identificadoras de las columnas a escalar
numeric_cols_to_scale = [col for col in numeric_cols if col not in categorical_numeric_cols]

print(f"\n📊 Columnas numéricas totales: {len(numeric_cols)}")
print(f"📊 Columnas categóricas/identificadoras (NO escaladas): {len(categorical_numeric_cols)}")
print(f"📊 Columnas numéricas a escalar: {len(numeric_cols_to_scale)}")

print(f"\n📋 Columnas categóricas/identificadoras excluidas del escalado:")
for col in categorical_numeric_cols:
    if col in numeric_cols:
        print(f"   - {col}")

print(f"\n📋 Columnas numéricas a escalar (incluyendo variables derivadas):")
print(f"   {numeric_cols_to_scale[:15]}...")  # Mostrar solo las primeras 15

# Crear escalador
scaler = StandardScaler()

# Aplicar escalado
print("\n🔧 Aplicando StandardScaler a variables numéricas...")
df_scaled = df_final.copy()

# Escalar solo las columnas numéricas continuas (incluyendo variables derivadas)
if len(numeric_cols_to_scale) > 0:
    df_scaled[numeric_cols_to_scale] = scaler.fit_transform(df_final[numeric_cols_to_scale])
    print(f"✅ Variables numéricas escaladas correctamente")
    
    # Mostrar estadísticas después del escalado
    print("\n📊 Estadísticas después del escalado:")
    print(df_scaled[numeric_cols_to_scale].describe().T[['mean', 'std', 'min', 'max']].head(15))
else:
    print("⚠️ No se encontraron columnas numéricas para escalar")

# Guardar el escalador para uso futuro
joblib.dump(scaler, MODELS_DIR / 'scaler.pkl')
print(f"\n✅ Escalador guardado en: {MODELS_DIR / 'scaler.pkl'}")

🔧 ESCALADO DE VARIABLES NUMÉRICAS

📊 Columnas numéricas totales: 20
📊 Columnas categóricas/identificadoras (NO escaladas): 4
📊 Columnas numéricas a escalar: 16

📋 Columnas categóricas/identificadoras excluidas del escalado:
   - CODIGO_DE_LA_ARL
   - AÑO_DE_INFORME
   - MES_DE_INFORME
   - ACTIVEC

📋 Columnas numéricas a escalar (incluyendo variables derivadas):
   ['RELA_DEP', 'RELA_INDEP', 'PRESUACCIDETRASUCE', 'MUERTES_REPOR_AT', 'NUEVAPENSIOINVA_R_AT', 'NUEVAPENSIOINVA_R_EL', 'INCAPERMAPARCIAR_AT', 'INCAPERMAPARCIAR_EL', 'TASA_ACCIDENTES_DEP', 'TASA_MUERTES_DEP', 'TASA_PENSIONES_ACCIDENTE', 'TASA_PENSIONES_ENFERMEDAD', 'TOTAL_PENSIONES', 'TOTAL_INCAPACIDADES', 'RATIO_INDEP_DEP']...

🔧 Aplicando StandardScaler a variables numéricas...
✅ Variables numéricas escaladas correctamente

📊 Estadísticas después del escalado:
                                   mean       std       min         max
RELA_DEP                   4.631357e-18  1.000008 -0.072296  146.052437
RELA_INDEP              

## 6. Exportación del Dataset Curado

### 6.1 Guardar Dataset Procesado

Exportamos el dataset curado en formato CSV y Parquet para su uso en los siguientes notebooks.

In [13]:
# Exportar dataset curado
print("=" * 80)
print("💾 EXPORTACIÓN DEL DATASET CURADO")
print("=" * 80)

# Guardar en formato CSV
csv_path = PROCESSED_DATA_DIR / 'dataset_curado.csv'
df_final.to_csv(csv_path, index=False)
print(f"\n✅ Dataset guardado en formato CSV: {csv_path}")

# Guardar en formato Parquet (más eficiente para grandes volúmenes de datos)
try:
    parquet_path = PROCESSED_DATA_DIR / 'dataset_curado.parquet'
    df_final.to_parquet(parquet_path, index=False)
    print(f"✅ Dataset guardado en formato Parquet: {parquet_path}")
except Exception as e:
    print(f"⚠️ No se pudo guardar en formato Parquet: {e}")

# Guardar los encoders
joblib.dump(label_encoders, MODELS_DIR / 'label_encoders.pkl')
print(f"✅ Label encoders guardados en: {MODELS_DIR / 'label_encoders.pkl'}")

# Mostrar información final
print("\n📊 INFORMACIÓN FINAL DEL DATASET:")
print(f"   - Dimensiones: {df_final.shape[0]} filas x {df_final.shape[1]} columnas")
print(f"   - Columnas originales: {len(df.columns)}")
print(f"   - Columnas nuevas: {len(df_final.columns) - len(df.columns)}")
print(f"   - Total de columnas: {len(df_final.columns)}")

print("\n📋 COLUMNAS FINALES:")
print(df_final.columns.tolist())

💾 EXPORTACIÓN DEL DATASET CURADO

✅ Dataset guardado en formato CSV: /Users/jualgozo/Documents/datoscol/DatosArl_Mac/data/processed/dataset_curado.csv
⚠️ No se pudo guardar en formato Parquet: Unable to find a usable engine; tried using: 'pyarrow', 'fastparquet'.
A suitable version of pyarrow or fastparquet is required for parquet support.
Trying to import the above resulted in these errors:
 - Missing optional dependency 'pyarrow'. pyarrow is required for parquet support. Use pip or conda to install pyarrow.
 - Missing optional dependency 'fastparquet'. fastparquet is required for parquet support. Use pip or conda to install fastparquet.
✅ Label encoders guardados en: /Users/jualgozo/Documents/datoscol/DatosArl_Mac/models/label_encoders.pkl

📊 INFORMACIÓN FINAL DEL DATASET:
   - Dimensiones: 61368 filas x 26 columnas
   - Columnas originales: 14
   - Columnas nuevas: 12
   - Total de columnas: 26

📋 COLUMNAS FINALES:
['DPTO', 'MPIO', 'CODIGO_DE_LA_ARL', 'AÑO_DE_INFORME', 'MES_DE_INFOR

### 6.2 Guardar Resumen del Preprocesamiento

Guardamos un resumen de los pasos de preprocesamiento realizados.

In [14]:
# Guardar resumen del preprocesamiento
from datetime import datetime

# Crear resumen
summary = f"""
# Resumen de Preprocesamiento - Riesgos Laborales Colombia
Fecha: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

## Dataset Original
- Dimensiones: {df.shape[0]} filas x {df.shape[1]} columnas
- Valores faltantes: {df.isnull().sum().sum()}
- Filas duplicadas: {df.duplicated().sum()}

## Dataset Curado
- Dimensiones: {df_final.shape[0]} filas x {df_final.shape[1]} columnas
- Valores faltantes: {df_final.isnull().sum().sum()}
- Filas duplicadas: {df_final.duplicated().sum()}

## Transformaciones Realizadas
1. **Limpieza de datos**:
   - Imputación de valores faltantes (mediana para numéricas, moda para categóricas)
   - Eliminación de filas duplicadas
   - Análisis de outliers (no eliminados, documentados)

2. **Codificación de variables categóricas**:
   - Label Encoding para: {categorical_cols}

3. **Escalado de variables numéricas**:
   - StandardScaler aplicado a {len(numeric_cols)} variables numéricas

4. **Variables derivadas creadas**:
   - TASA_ACCIDENTES_DEP: Tasa de accidentes por trabajador dependiente
   - TASA_MUERTES_DEP: Tasa de muertes por trabajador dependiente
   - TASA_PENSIONES_ACCIDENTE: Tasa de pensiones por accidente
   - TASA_PENSIONES_ENFERMEDAD: Tasa de pensiones por enfermedad
   - TOTAL_PENSIONES: Total de pensiones (accidentes + enfermedades)
   - TOTAL_INCAPACIDADES: Total de incapacidades (accidentes + enfermedades)
   - RATIO_INDEP_DEP: Ratio de trabajadores independientes vs dependientes
   - TOTAL_TRABAJADORES: Total de trabajadores (dependientes + independientes)

## Archivos Generados
- dataset_curado.csv: Dataset curado en formato CSV
- dataset_curado.parquet: Dataset curado en formato Parquet
- label_encoders.pkl: Diccionario de LabelEncoders para variables categóricas
- scaler.pkl: StandardScaler para variables numéricas

## Próximos Pasos
1. Implementar modelos de predicción (notebook 03_Modelos_Prediccion.ipynb)
2. Analizar patrones y anomalías (notebook 04_Analisis_Patrones.ipynb)
3. Generar explicabilidad de modelos (notebook 05_Explicabilidad_Resultados.ipynb)
"""

# Guardar resumen
with open(REPORTS_DIR / 'resumen_preprocesamiento.md', 'w', encoding='utf-8') as f:
    f.write(summary)

print(f"✅ Resumen guardado en: {REPORTS_DIR / 'resumen_preprocesamiento.md'}")

✅ Resumen guardado en: /Users/jualgozo/Documents/datoscol/DatosArl_Mac/reports/resumen_preprocesamiento.md


## 7. Conclusión

Este notebook ha realizado el preprocesamiento completo del dataset de riesgos laborales. Los pasos realizados incluyen:

1. **Limpieza de datos**:
   - Imputación de valores faltantes
   - Eliminación de duplicados
   - Análisis de outliers

2. **Transformación de variables**:
   - Codificación de variables categóricas con Label Encoding
   - Escalado de variables numéricas con StandardScaler

3. **Creación de variables derivadas**:
   - Tasas de accidentes, muertes y pensiones
   - Ratios de trabajadores
   - Totales de pensiones e incapacidades

4. **Exportación del dataset curado**:
   - Formato CSV y Parquet
   - Guardado de encoders y scalers para uso futuro

El dataset curado está listo para ser utilizado en los siguientes notebooks:
- `03_Modelos_Prediccion.ipynb`: Implementación de modelos de predicción
- `04_Analisis_Patrones.ipynb`: Análisis de patrones y anomalías
- `05_Explicabilidad_Resultados.ipynb`: Explicabilidad de modelos

---

**Nota**: Este notebook debe ser ejecutado después de `01_Exploracion_Datos.ipynb` y antes de los notebooks de modelado.